# DS6 — Information Theory (InT) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**
**Dataset:** DS6 (Danish Adolescents, quasi-experimental, N=243 behavioral)
**Model family:** Information Theory (InT)
**Date:** 2026-08-06
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds6_int_blueprint.md`

---

## Scope and purpose

Third DS6 CAS model. Depends on StoP (73, for sig_hat and the daily N_t series) and NeT
(74, for H_trans) -- both complete. Computes entropy dimensions per study_period, bridges to
both prior models, and adds one pooled mutual-information test motivated directly by NeT's
finding.

- **Goal:** Quantify predictability/diversity of DS6 behavior (app choice, timing, daily
  volume) per period; verify the StoP-InT and NeT-InT bridges; test whether study_period
  reduces uncertainty about app choice.
- **Not goal:** Phenotype classification, personalization, cross-dataset harmonization.
- **Output:** Per-participant entropy profile (H_app/H_hour/H_emp/H_SDE/I_consec, both
  periods), condition x period summary, NeT bridge validity check, one pooled MI result,
  psychosocial correlations.

### FLAG decisions in effect (blueprint Part 0, 2, 4)

| Flag | Decision |
|---|---|
| FLAG InT-1 | H_emp uses K=10 QUANTILE bins per participant (not DS5's fixed [0,1] bins -- DS6's N_t is an unbounded count, 1-333/day) |
| FLAG InT-2 | H_out excluded entirely (DS6 has no outcome variance -- would be a constant 0, not a measurement) |
| FLAG InT-3 | New pooled MI(study_period; app_name) test -- DS5 deferred all MI/KL; DS6 has a specific reason (NeT's already-completed HHI_net-period finding) |

### WARNING — proxy status

All entropy measures describe patterns in observed event distributions, not directly
observed psychological states (predictability, self-control, or attentional focus). A user
with low H_app is routine in WHICH app they open; this is not evidence of low urge or high
self-control. This distinction must be maintained throughout interpretation.

---

## Section 1: Pure Theory — Information Theory in CAS (paper Section 2.5)

> "Entropy can be computed for target app, resolution code, time of day, condition-specific
> outcomes, app category, or user-day response states."
>
> "Mutual information can estimate how strongly condition, target app, prior outcome, or
> time reduces uncertainty about opening versus dismissal... Time of day may reduce
> uncertainty if nighttime attempts are more likely to be opened than daytime attempts."

DS6 cannot use "resolution code" / "condition-specific outcomes" / MI-toward-outcome (no
outcome variance — see StoP blueprint Part 0). Uses the paper's other named measures:
target app (H_app), time of day (H_hour), user-day response states (H_emp, on StoP's N_t).

### Core equations

    Shannon entropy:        H(X) = -sum_x p(x) * log2(p(x))
    Gaussian diff. entropy:  H_SDE = 0.5 * log2(2*pi*e*sigma^2)     (bridge from StoP sig_hat)
    Mutual information:      I(X;Y) = H(X) - H(X|Y) = H(X) + H(Y) - H(X,Y)
    NeT-InT bridge:           I_consec = H_app - H_trans             (>= 0 required)

All measures computed separately for study_period in {baseline, intervention}, matching
StoP (73) and NeT (74).

In [ ]:
# =============================================================================
# Imports and paths
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from datetime import datetime

ROOT     = Path('..').resolve()
CLEANED  = ROOT / 'outputs' / 'cleaned'
FEAT     = ROOT / 'outputs' / 'features' / 'ds6'
STOP_DIR = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'stochastic_processes'
NET_DIR  = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'network_models'
REPORTS  = ROOT / 'outputs' / 'model_outputs' / 'ds6' / 'information_theory'
FIGURES  = REPORTS / 'figures'
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

K_BINS = 10   # FLAG InT-1: quantile bins, per participant

MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print('REPORTS :', REPORTS)
print('Run at  :', MODEL_RUN_TS)

In [ ]:
# =============================================================================
# Load data
# =============================================================================
beh = pd.read_csv(CLEANED / 'ds6_behavioral_clean.csv', low_memory=False)
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp_utc'], utc=True)

beh_feat  = pd.read_csv(FEAT / 'ds6_behavior_features.csv')
survey    = pd.read_csv(FEAT / 'ds6_survey_features.csv')
stop_params = pd.read_csv(STOP_DIR / 'participant_stop_parameters.csv')
daily_long  = pd.read_csv(STOP_DIR / 'daily_series_long.csv', parse_dates=['event_date'])
net_params  = pd.read_csv(NET_DIR / 'participant_network_metrics.csv')

print(f'Behavioral rows : {len(beh):,}')
print(f'StoP params     : {len(stop_params)} participants')
print(f'NeT params      : {len(net_params)} participants')
print(f'Daily series    : {len(daily_long):,} rows')

---

## Section 2: H_app and H_hour (per period)

In [ ]:
# =============================================================================
# H_app -- re-framed Stage 1 features (same precedent as DS5's InT notebook)
# =============================================================================
h_app_df = beh_feat[['participant_id_canonical', 'app_entropy_baseline', 'app_entropy_intervention']].rename(
    columns={'app_entropy_baseline': 'H_app_baseline', 'app_entropy_intervention': 'H_app_intervention'}
)
print('H_app (from Stage 1 app_entropy_baseline/intervention):')
print(h_app_df[['H_app_baseline', 'H_app_intervention']].describe().round(4))

In [ ]:
# =============================================================================
# H_hour -- NOT pre-split by period in Stage 1; computed fresh here
# =============================================================================
def shannon_entropy(series):
    counts = series.value_counts()
    total = counts.sum()
    if total == 0:
        return np.nan
    p = counts / total
    return float(-(p * np.log2(p)).sum())

h_hour_records = []
for pid, grp in beh.groupby('participant_id_canonical'):
    row = {'participant_id_canonical': pid}
    for period in ['baseline', 'intervention']:
        sub = grp[grp['study_period'] == period]
        row[f'H_hour_{period}'] = shannon_entropy(sub['hour_of_day']) if len(sub) >= 5 else np.nan
    h_hour_records.append(row)
h_hour_df = pd.DataFrame(h_hour_records)
print('H_hour (computed fresh per period):')
print(h_hour_df[['H_hour_baseline', 'H_hour_intervention']].describe().round(4))

---

## Section 3: H_emp -- empirical entropy of the binned daily N_t series (FLAG InT-1)

In [ ]:
# =============================================================================
# H_emp: K=10 QUANTILE bins per participant (NOT DS5's fixed [0,1] bins)
# =============================================================================
def empirical_entropy_quantile(x_values, k=K_BINS):
    x = np.asarray(x_values, dtype=float)
    if len(x) < k or np.all(x == x[0]):
        return np.nan, np.nan
    try:
        bins = pd.qcut(x, q=k, duplicates='drop')
    except ValueError:
        return np.nan, np.nan
    counts = bins.value_counts()
    total = counts.sum()
    p = counts / total
    p = p[p > 0]
    h = float(-(p * np.log2(p)).sum())
    return h, len(counts)  # entropy, effective K

h_emp_records = []
for (pid, period), grp in daily_long.groupby(['participant_id_canonical', 'study_period']):
    h, eff_k = empirical_entropy_quantile(grp['N_t'].values)
    h_emp_records.append(dict(participant_id_canonical=pid, study_period=period, H_emp=h, effective_K=eff_k))

h_emp_long = pd.DataFrame(h_emp_records)
h_emp_df = h_emp_long.pivot(index='participant_id_canonical', columns='study_period', values='H_emp')
h_emp_df.columns = [f'H_emp_{c}' for c in h_emp_df.columns]
h_emp_df = h_emp_df.reset_index()

print('H_emp (K=10 quantile bins per participant):')
print(h_emp_df[['H_emp_baseline', 'H_emp_intervention']].describe().round(4))
print(f'\nMax theoretical H_emp (K=10 uniform): {np.log2(K_BINS):.4f} bits')
print(f'\nEffective K distribution (baseline):')
print(h_emp_long[h_emp_long.study_period=='baseline']['effective_K'].describe())

---

## Section 4: H_SDE -- Gaussian differential entropy bridge from StoP

In [ ]:
# =============================================================================
# H_SDE = 0.5 * log2(2*pi*e*sig_hat^2)  -- same bridge equation as DS5 (paper gap, not
# yet formalized in the conceptual paper -- same caveat DS5's InT notebook recorded)
# =============================================================================
h_sde_df = stop_params[['participant_id_canonical', 'sig_hat_baseline', 'sig_hat_intervention']].copy()
h_sde_df['H_SDE_baseline'] = 0.5 * np.log2(2.0 * np.pi * np.e * h_sde_df['sig_hat_baseline']**2)
h_sde_df['H_SDE_intervention'] = 0.5 * np.log2(2.0 * np.pi * np.e * h_sde_df['sig_hat_intervention']**2)

print('H_SDE (bits):')
print(h_sde_df[['H_SDE_baseline', 'H_SDE_intervention']].describe().round(4))

---

## Section 5: Assemble entropy profile + StoP-InT bridge check (H_SDE vs H_emp)

In [ ]:
# =============================================================================
# Assemble full entropy profile
# =============================================================================
entropy_df = h_app_df.merge(h_hour_df, on='participant_id_canonical', how='outer') \
                      .merge(h_emp_df, on='participant_id_canonical', how='outer') \
                      .merge(h_sde_df.drop(columns=['sig_hat_baseline', 'sig_hat_intervention']),
                             on='participant_id_canonical', how='outer')
cond_lookup = beh.groupby('participant_id_canonical')['study_condition'].agg(lambda s: s.mode().iloc[0])
entropy_df['study_condition'] = entropy_df['participant_id_canonical'].map(cond_lookup)

print(f'Entropy profile: {len(entropy_df)} participants, {entropy_df.shape[1]} columns')

# StoP-InT bridge check: H_SDE should correlate with H_emp (both derived from the same
# underlying variability, via different routes)
for period in ['baseline', 'intervention']:
    sub = entropy_df[[f'H_SDE_{period}', f'H_emp_{period}']].dropna()
    r, p = stats.pearsonr(sub[f'H_SDE_{period}'], sub[f'H_emp_{period}'])
    print(f'{period}: r(H_SDE, H_emp) = {r:.4f}, p={p:.4g}, n={len(sub)}')

---

## Section 6: NeT-InT Bridge -- I_consec = H_app - H_trans (mandatory sanity gate)

**Bug found and fixed here (2026-08-06), same class as Structured's NeT-InT bridge bug --
found in TWO successive attempts, both caught by the same mandatory assertion.**

*Attempt 1:* used Stage 1's `app_entropy_baseline`/`app_entropy_intervention` directly as
H_app. Produced 2 (baseline) / 3 (intervention) participants with slightly negative
I_consec (min -0.0014). Root cause: Stage 1's `app_entropy` counts ALL events in the
period, including each participant's last event (which has no outgoing transition and is
therefore absent from NeT's transition-edge marginals) -- a small boundary mismatch.

*Attempt 2:* fixed by computing H_app from the transition edge list's `app_from` marginal
(the distribution NeT's own `flow_share` already uses for HHI_net). This REDUCED but did
not eliminate violations (1 baseline participant, now with a LARGER violation, -0.0493) --
the assertion caught this second mistake too. Root cause: the correct identity is
`I(X_t; X_t+1) = H(X_t+1) - H(X_t+1 | X_t)`, so the H(X_t+1) term must be the **destination**
("app_to") marginal, not the source ("app_from") marginal used in attempt 2. For most
participants (long chains) the two marginals nearly coincide, masking the error; for
short chains (this violator had only M=43 transitions) the boundary difference is large
enough to flip the sign. Verified by hand: for the violating participant, `app_from`
marginal H=0.910 vs H_trans=0.960 (I_consec=-0.049, invalid); `app_to` marginal H=1.032 vs
same H_trans (I_consec=+0.072, valid). This is exactly the fix already recorded in project
memory for Structured's own version of this bug ("target-state marginal from NeT's own
edge list") -- should have been applied directly instead of re-deriving from `app_from`
first.

*Final fix (this cell uses it):* H_app_marginal = entropy of the `app_to` column from the
transition edge list, per participant per period.

In [ ]:
# =============================================================================
# H_app_marginal: entropy of the app_to (DESTINATION) distribution FROM THE TRANSITION
# EDGES (NOT Stage 1's app_entropy, NOT the app_from/source marginal either -- see
# markdown cell above for why app_from was tried first and also failed the bridge gate).
# The identity I(X_t;X_t+1) = H(X_t+1) - H(X_t+1|X_t) requires H(X_t+1) to be the
# NEXT-state marginal, i.e. app_to, not app_from.
# =============================================================================
edge_list = pd.read_csv(NET_DIR / 'transition_edge_list.csv')

h_app_marginal_records = []
for (pid, period), grp in edge_list.groupby(['participant_id_canonical', 'study_period']):
    to_counts = grp.groupby('app_to')['count'].sum()
    h_app_marginal_records.append(dict(
        participant_id_canonical=pid, study_period=period,
        H_app_marginal=shannon_entropy(pd.Series(np.repeat(to_counts.index, to_counts.values)))
    ))
h_app_marginal_long = pd.DataFrame(h_app_marginal_records)
h_app_marginal_df = h_app_marginal_long.pivot(
    index='participant_id_canonical', columns='study_period', values='H_app_marginal'
)
h_app_marginal_df.columns = [f'H_app_marginal_{c}' for c in h_app_marginal_df.columns]
h_app_marginal_df = h_app_marginal_df.reset_index()

print('H_app_marginal (transition-consistent, used for I_consec bridge):')
print(h_app_marginal_df[['H_app_marginal_baseline', 'H_app_marginal_intervention']].describe().round(4))

In [ ]:
# =============================================================================
# I_consec = H_app_marginal - H_trans, per period. MUST be >= 0 (mutual information
# cannot be negative) -- this exact check caught a real bug in Structured's own
# NeT-InT bridge, and now this one too (see markdown cell above).
# =============================================================================
bridge = entropy_df.merge(h_app_marginal_df, on='participant_id_canonical', how='left').merge(
    net_params[['participant_id_canonical', 'H_trans_baseline', 'H_trans_intervention']],
    on='participant_id_canonical', how='left'
)
bridge['I_consec_baseline'] = bridge['H_app_marginal_baseline'] - bridge['H_trans_baseline']
bridge['I_consec_intervention'] = bridge['H_app_marginal_intervention'] - bridge['H_trans_intervention']

for period in ['baseline', 'intervention']:
    col = f'I_consec_{period}'
    n_valid = bridge[col].notna().sum()
    n_neg = (bridge[col] < -1e-9).sum()
    print(f'{period}: I_consec valid={n_valid}, negative (BUG if >0)={n_neg}')
    print(bridge[col].describe().round(4))
    print()

assert (bridge['I_consec_baseline'].dropna() >= -1e-9).all(), 'NeT-InT bridge violated (baseline)!'
assert (bridge['I_consec_intervention'].dropna() >= -1e-9).all(), 'NeT-InT bridge violated (intervention)!'
print('NeT-InT bridge check PASSED: I_consec >= 0 for all participants, both periods.')

entropy_df = bridge

---

## Section 7: Mutual Information -- does study_period reduce uncertainty about app choice? (FLAG InT-3)

In [ ]:
# =============================================================================
# Pooled MI(study_period; app_name), computed on the full event table (not per-participant)
# =============================================================================
def mutual_information(df, col_x, col_y):
    joint = pd.crosstab(df[col_x], df[col_y], normalize=True)
    px = joint.sum(axis=1)
    py = joint.sum(axis=0)
    mi = 0.0
    for i in joint.index:
        for j in joint.columns:
            pij = joint.loc[i, j]
            if pij > 0:
                mi += pij * np.log2(pij / (px[i] * py[j]))
    return float(mi)

I_period_app = mutual_information(beh, 'study_period', 'app_name')
H_app_pooled = shannon_entropy(beh['app_name'])
norm_mi = I_period_app / H_app_pooled if H_app_pooled > 0 else np.nan

print(f'H(app), pooled                 : {H_app_pooled:.4f} bits')
print(f'I(study_period; app_name)      : {I_period_app:.4f} bits')
print(f'Normalized MI (I / H(app))     : {norm_mi:.4f}')
print(f'\nstudy_period distribution (pooled):')
print(beh['study_period'].value_counts(normalize=True).round(3))

mutual_information_summary = pd.DataFrame([dict(
    pair='study_period;app_name', H_app_pooled=H_app_pooled,
    I_mutual_info=I_period_app, normalized_mi=norm_mi, n_events=len(beh),
)])

---

## Section 8: Condition x Period Summary and Psychosocial Correlates

In [ ]:
# =============================================================================
# Condition x period entropy summary
# =============================================================================
measures = ['H_app', 'H_hour', 'H_emp', 'H_SDE', 'I_consec']
cond_period_rows = []
for cond in ['control', 'default', 'intervention']:
    for period in ['baseline', 'intervention']:
        sub = entropy_df[entropy_df['study_condition'] == cond]
        row = dict(study_condition=cond, study_period=period, n=len(sub))
        for m in measures:
            col = f'{m}_{period}'
            vals = sub[col].dropna()
            row[f'{m}_mean'] = vals.mean()
            row[f'{m}_median'] = vals.median()
        cond_period_rows.append(row)
condition_period_summary = pd.DataFrame(cond_period_rows).round(4)
print(condition_period_summary.to_string(index=False))

In [ ]:
# =============================================================================
# Psychosocial correlations (exploratory)
# =============================================================================
merged_psych = entropy_df.merge(
    survey[['participant_id_canonical', 'addiction_index_pre', 'self_control_index_pre']],
    on='participant_id_canonical', how='left'
)
entropy_cols = [f'{m}_{p}' for m in measures for p in ['baseline', 'intervention']] + \
               ['H_app_marginal_baseline', 'H_app_marginal_intervention']
psych_cols = ['addiction_index_pre', 'self_control_index_pre']

corr_records = []
for ecol in entropy_cols:
    for pcol in psych_cols:
        sub = merged_psych[[ecol, pcol]].dropna()
        if len(sub) >= 10:
            r, p = stats.pearsonr(sub[ecol], sub[pcol])
            corr_records.append(dict(entropy_measure=ecol, psychosocial=pcol, n=len(sub), r=round(r,4), p=round(p,4)))
psychosocial_entropy_correlations = pd.DataFrame(corr_records)
print(psychosocial_entropy_correlations.sort_values('p').to_string(index=False))

In [ ]:
# =============================================================================
# Figure 1: entropy measures by period
# =============================================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, m in zip(axes, ['H_app', 'H_hour', 'H_emp', 'H_SDE']):
    data = [entropy_df[f'{m}_baseline'].dropna(), entropy_df[f'{m}_intervention'].dropna()]
    ax.violinplot(data, showmeans=True)
    ax.set_xticks([1, 2]); ax.set_xticklabels(['baseline', 'intervention'])
    ax.set_title(m)
plt.suptitle('DS6 InT -- entropy measures by period')
plt.tight_layout()
fig.savefig(FIGURES / 'fig1_int_entropy_by_period.png', dpi=150)
plt.show()
print('Saved fig1_int_entropy_by_period.png')

In [ ]:
# =============================================================================
# Figure 2: H_SDE vs H_emp bridge check
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, period in zip(axes, ['baseline', 'intervention']):
    sub = entropy_df[[f'H_SDE_{period}', f'H_emp_{period}']].dropna()
    ax.scatter(sub[f'H_SDE_{period}'], sub[f'H_emp_{period}'], alpha=0.5, s=20)
    r, p = stats.pearsonr(sub[f'H_SDE_{period}'], sub[f'H_emp_{period}'])
    ax.set_xlabel('H_SDE (Gaussian bridge from sig_hat)')
    ax.set_ylabel('H_emp (empirical, quantile-binned N_t)')
    ax.set_title(f'{period}: r={r:.3f}, p={p:.4g}, n={len(sub)}')
plt.tight_layout()
fig.savefig(FIGURES / 'fig2_int_hsde_hemp_bridge.png', dpi=150)
plt.show()
print('Saved fig2_int_hsde_hemp_bridge.png')

In [ ]:
# =============================================================================
# Figure 3: NeT-InT bridge scatter (I_consec components)
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, period in zip(axes, ['baseline', 'intervention']):
    # BUG FIX (2026-08-07): entropy_df is assembled entirely via .merge(on='participant_id_canonical')
    # calls (Section 5/6), so it keeps a default RangeIndex -- participant_id_canonical is a plain
    # column, not the index. The original line joined entropy_df's subset (RangeIndex 0..242) against
    # net_params.set_index('participant_id_canonical') (a string-valued index) via .join(), which joins
    # on index by default -- an inner join between an integer RangeIndex and a string ID index matches
    # zero rows, silently producing an empty dataframe and a blank scatter plot for both panels. Fixed
    # by also set_index-ing the left side on participant_id_canonical before joining, matching what the
    # right side already did. Does not touch H_app_marginal, H_trans, I_consec, or the bridge-check gate
    # -- entropy_df and net_params themselves are unchanged; only how this cell's local `sub` is built.
    sub = entropy_df.set_index('participant_id_canonical')[[f'H_app_marginal_{period}']].join(
        net_params.set_index('participant_id_canonical')[[f'H_trans_{period}']],
        how='inner'
    )
    ax.scatter(sub[f'H_trans_{period}'], sub[f'H_app_marginal_{period}'], alpha=0.5, s=20)
    lims = [min(sub.min()), max(sub.max())]
    ax.plot(lims, lims, color='gray', linewidth=0.8, linestyle='--', label='H_app_marginal = H_trans (I_consec=0)')
    ax.set_xlabel('H_trans (NeT, notebook 74)')
    ax.set_ylabel('H_app_marginal (transition-consistent, this notebook)')
    ax.set_title(f'{period}: NeT-InT bridge (points above line: I_consec > 0)')
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES / 'fig3_int_net_bridge_scatter.png', dpi=150)
plt.show()
print('Saved fig3_int_net_bridge_scatter.png')

In [ ]:
# =============================================================================
# Figure 4: condition x period comparison (H_app)
# =============================================================================
fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
conds = ['control', 'default', 'intervention']
x = np.arange(len(conds))
base_h = [condition_period_summary.query("study_condition==@c and study_period=='baseline'")['H_app_mean'].values[0] for c in conds]
intv_h = [condition_period_summary.query("study_condition==@c and study_period=='intervention'")['H_app_mean'].values[0] for c in conds]
ax.bar(x - width/2, base_h, width, label='baseline', color='#4C72B0')
ax.bar(x + width/2, intv_h, width, label='intervention', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(conds)
ax.set_ylabel('mean H_app')
ax.set_title('DS6 InT -- mean app entropy by study_condition x study_period')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / 'fig4_int_condition_period_comparison.png', dpi=150)
plt.show()
print('Saved fig4_int_condition_period_comparison.png')

In [ ]:
# =============================================================================
# Export all outputs
# =============================================================================
export_cols = ['participant_id_canonical', 'study_condition',
               'H_app_baseline', 'H_app_intervention',
               'H_app_marginal_baseline', 'H_app_marginal_intervention',
               'H_hour_baseline', 'H_hour_intervention',
               'H_emp_baseline', 'H_emp_intervention',
               'H_SDE_baseline', 'H_SDE_intervention',
               'I_consec_baseline', 'I_consec_intervention']
participant_entropy_metrics = entropy_df[export_cols].copy()
participant_entropy_metrics.to_csv(REPORTS / 'participant_entropy_metrics.csv', index=False)
print(f'Saved participant_entropy_metrics.csv ({len(participant_entropy_metrics)} rows)')

condition_period_summary.to_csv(REPORTS / 'condition_period_summary.csv', index=False)
print(f'Saved condition_period_summary.csv ({len(condition_period_summary)} rows)')

net_int_bridge_check = entropy_df[['participant_id_canonical', 'H_app_marginal_baseline', 'H_app_marginal_intervention',
                                     'I_consec_baseline', 'I_consec_intervention']].merge(
    net_params[['participant_id_canonical', 'H_trans_baseline', 'H_trans_intervention']],
    on='participant_id_canonical', how='left'
)
# NOTE (audit fix, 2026-08-06): use pd.NA / boolean masking so rows where I_consec is
# NaN (NeT-ineligible participant) are stored as missing, not False -- NaN >= -1e-9
# evaluates to False in pandas, which would otherwise misrepresent "not applicable" as
# "bridge violated" in the saved CSV. The underlying assertion above already handled this
# correctly by filtering on the null mask; this fixes the same issue in the EXPORTED file.
net_int_bridge_check['bridge_valid_baseline'] = np.where(
    net_int_bridge_check['I_consec_baseline'].notna(),
    net_int_bridge_check['I_consec_baseline'] >= -1e-9, np.nan)
net_int_bridge_check['bridge_valid_intervention'] = np.where(
    net_int_bridge_check['I_consec_intervention'].notna(),
    net_int_bridge_check['I_consec_intervention'] >= -1e-9, np.nan)
net_int_bridge_check.to_csv(REPORTS / 'net_int_bridge_check.csv', index=False)
print(f'Saved net_int_bridge_check.csv ({len(net_int_bridge_check)} rows)')

mutual_information_summary.to_csv(REPORTS / 'mutual_information_summary.csv', index=False)
print(f'Saved mutual_information_summary.csv ({len(mutual_information_summary)} rows)')

psychosocial_entropy_correlations.to_csv(REPORTS / 'psychosocial_entropy_correlations.csv', index=False)
print(f'Saved psychosocial_entropy_correlations.csv ({len(psychosocial_entropy_correlations)} rows)')

In [ ]:
# =============================================================================
# model_run_summary.md
# =============================================================================
r_base, p_base = stats.pearsonr(*entropy_df[['H_SDE_baseline','H_emp_baseline']].dropna().values.T)
r_intv, p_intv = stats.pearsonr(*entropy_df[['H_SDE_intervention','H_emp_intervention']].dropna().values.T)

summary_md = f"""# DS6 InT -- Model Run Summary

**Run at:** {MODEL_RUN_TS}
**Notebook:** notebooks/75_ds6_information_theory.ipynb
**Blueprint:** outputs/reports/stage4/step1_modeling/ds6_int_blueprint.md

## Key results
- Mean H_app (baseline / intervention): {entropy_df['H_app_baseline'].mean():.4f} / {entropy_df['H_app_intervention'].mean():.4f} bits
- Mean H_hour (baseline / intervention): {entropy_df['H_hour_baseline'].mean():.4f} / {entropy_df['H_hour_intervention'].mean():.4f} bits
- Mean H_emp (baseline / intervention): {entropy_df['H_emp_baseline'].mean():.4f} / {entropy_df['H_emp_intervention'].mean():.4f} bits
- Mean H_SDE (baseline / intervention): {entropy_df['H_SDE_baseline'].mean():.4f} / {entropy_df['H_SDE_intervention'].mean():.4f} bits
- StoP-InT bridge r(H_SDE,H_emp): baseline r={r_base:.4f} p={p_base:.4g}; intervention r={r_intv:.4f} p={p_intv:.4g}
- NeT-InT bridge: I_consec >= 0 for ALL participants, both periods (PASSED, mandatory gate)
- Mean I_consec (baseline / intervention): {entropy_df['I_consec_baseline'].mean():.4f} / {entropy_df['I_consec_intervention'].mean():.4f} bits
- Pooled MI(study_period; app_name) = {I_period_app:.4f} bits (normalized: {norm_mi:.4f}), H(app)={H_app_pooled:.4f} bits

## Proxy warning
Entropy measures describe patterns in observed event distributions (which app, what hour,
how much daily volume), not directly observed psychological states. Low H_app means routine
in app CHOICE, not evidence of low urge or high self-control.

## Known limitations
- H_out excluded entirely -- DS6 has no dismiss/close outcome variance (see StoP blueprint Part 0).
- H_SDE bridge equation (Gaussian differential entropy) not yet formalized in the conceptual paper -- same gap DS5's InT notebook recorded.
- H_emp uses per-participant quantile bins (FLAG InT-1), not DS5's fixed [0,1] range -- H_emp values are not directly comparable in absolute terms to DS5's H_emp (different variable, different binning).
- Psychosocial correlations are exploratory, uncorrected for multiple comparisons ({len(psychosocial_entropy_correlations)} tests).
"""

(REPORTS / 'model_run_summary.md').write_text(summary_md)
print(summary_md)

---

## Section 9: Interpretation and Limits

### Headline findings

1. **StoP-InT bridge supports the Gaussian-increment assumption:** r(H_SDE, H_emp) = 0.583
   (baseline, p=6.9e-23) / 0.719 (intervention, p=6.7e-39) -- both strong, highly
   significant. Evidence the constant-drift Gaussian-increment assumption behind StoP's
   estimator is not badly violated for DS6's count process.
2. **Pooled MI(study_period; app_name) is a genuine null result** (0.0003 bits, normalized
   0.02%). Does NOT contradict NeT's per-participant HHI_net-volume correlation --
   individuals can each narrow toward their OWN favorite app without the population-level
   app distribution shifting in a way period predicts. Different apps are different
   participants' favorites; both results are correct and reported as-is.

### What the InT model reveals

| Result | Interpretation |
|---|---|
| H_app, H_hour (baseline vs intervention) | Whether app-choice and timing diversity shift once the intervention period begins |
| H_emp | Predictability of daily attempt VOLUME (not choice) -- bridges to StoP's sig_hat |
| H_SDE vs H_emp | Whether the Gaussian-increment assumption behind StoP's method-of-moments estimator is empirically reasonable |
| I_consec (NeT-InT bridge) | How much of app-choice uncertainty is explained by knowing the immediately preceding app |
| I(study_period; app) | Whether the intervention period shifts the POOLED app distribution, not just individual entropy |

### What this model does NOT show

- A causal account of why entropy differs by period -- purely descriptive/associational.
- Psychological predictability, self-control, or urge -- only observed-distribution structure.
- Comparability of DS6's H_emp to DS5's H_emp in absolute bit terms (different state variable, different binning scheme -- FLAG InT-1).

### Forward connections (informational only -- not implemented here)

- **InT -> Step 2 phenotyping:** H_app/H_emp/H_SDE/I_consec per period, plus the pooled MI
  result, are candidate phenotype dimensions once DS6 Step 1 progresses further (DyS, SyD,
  StM remain).
- **InT -> StM:** entropy-based macrostate framing (if StM is eventually built for DS6,
  matching how DS5/Structured's StM notebooks cross-checked against their own InT marginals).